# W2-D2: Root Cause Analysis — Graph + Retrieval Pipeline

Pipeline gồm 3 tầng:
1. **Graph RCA** — PageRank trên reverse graph + timestamp scoring
2. **Retrieval** — tìm incident lịch sử tương tự (keyword similarity)
3. **Classifier** — lấy `class` + `actions` từ top-1 similar incident (kNN-style)

In [53]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import json
import networkx as nx
from datetime import datetime
from pathlib import Path

print('Libraries loaded OK')
print(f'networkx version: {nx.__version__}')

Libraries loaded OK
networkx version: 3.3


In [54]:
# ── Cell 2: Load dữ liệu ─────────────────────────────────────────────────────

# Cluster summary từ W2D1
with open(r'D:\Xbrain\Phase 2 AIops\w2\d1\results\cluster_summary.json') as f:
    cluster_data = json.load(f)
clusters = cluster_data['clusters']

# Alerts raw
alerts = []
with open(r'D:\Xbrain\Phase 2 AIops\w2\d2\dataset\alerts_sample.jsonl') as f:
    for line in f:
        line = line.strip()
        if line:
            alerts.append(json.loads(line))

# Service graph
with open(r'D:\Xbrain\Phase 2 AIops\w2\d2\dataset\services.json') as f:
    services_data = json.load(f)

# Incident history
with open(r'D:\Xbrain\Phase 2 AIops\w2\d2\dataset\incidents_history.json') as f:
    history_data = json.load(f)
incidents = history_data['incidents']

print(f'Clusters loaded   : {len(clusters)}')
print(f'Alerts loaded     : {len(alerts)}')
print(f'Services loaded   : {len(services_data["services"])} services, {len(services_data["edges"])} edges')
print(f'Incidents loaded  : {len(incidents)}')

Clusters loaded   : 3
Alerts loaded     : 20
Services loaded   : 10 services, 17 edges
Incidents loaded  : 29


In [55]:
# ── Cell 3: Build service graph từ services.json ──────────────────────────────

# Chỉ dùng service-to-service edges (bỏ qua edge tới DB/Redis/Kafka)
service_names = {s['name'] for s in services_data['services']}

graph = nx.DiGraph()
graph.add_nodes_from(service_names)

for edge in services_data['edges']:
    src, dst = edge['from'], edge['to']
    # Chỉ thêm edge giữa service (không phải store)
    if src in service_names and dst in service_names:
        graph.add_edge(src, dst)

print(f'Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
print('\nEdges:')
for u, v in graph.edges():
    print(f'  {u} → {v}')

Graph: 10 nodes, 10 edges

Edges:
  catalog-svc → recommender-svc
  cart-svc → catalog-svc
  checkout-svc → cart-svc
  checkout-svc → payment-svc
  checkout-svc → inventory-svc
  checkout-svc → notification-svc
  edge-lb → auth-svc
  edge-lb → catalog-svc
  edge-lb → search-svc
  edge-lb → checkout-svc


In [56]:
# ── Cell 4: Helper — build alert lookup theo service ─────────────────────────

# Map: service → list of alert timestamps
service_alert_times = {}
for alert in alerts:
    svc = alert['service']
    ts = datetime.fromisoformat(alert['ts'].replace('Z', '+00:00'))
    if svc not in service_alert_times:
        service_alert_times[svc] = []
    service_alert_times[svc].append(ts)

# Lấy timestamp đầu tiên (sớm nhất) của mỗi service
service_first_alert = {svc: min(times) for svc, times in service_alert_times.items()}

print('Service → first alert time:')
for svc, ts in sorted(service_first_alert.items(), key=lambda x: x[1]):
    print(f'  {svc:20s} {ts.strftime("%H:%M:%S")}')

Service → first alert time:
  payment-svc          09:42:01
  checkout-svc         09:42:45
  edge-lb              09:43:15
  cart-svc             09:43:32
  notification-svc     09:43:50
  recommender-svc      09:45:10
  search-svc           09:46:50


In [57]:
# ── Cell 5: Graph RCA function ────────────────────────────────────────────────

def graph_rca(cluster, graph, service_first_alert, top_k=3):
    """
    Input : cluster dict, full service graph, service→first_alert_time
    Output: list of [service, combined_score] sorted desc
    """
    alert_services = set(cluster['services'])
    
    # Subgraph chỉ gồm service đang alert
    subgraph = graph.subgraph(alert_services).copy()
    
    # Nếu chỉ có 1 service → trả về luôn
    if len(alert_services) == 1:
        svc = list(alert_services)[0]
        return [[svc, 1.0]]
    
    # ── PageRank trên graph gốc ───────────────────────────────────────────────
    # Graph gốc: A→B nghĩa là A gọi B → callee (B) được nhiều node trỏ vào
    # KHÔNG dùng reverse: reverse làm edge-lb score cao nhất (victim, không phải culprit)
    pagerank_scores = nx.pagerank(subgraph, alpha=0.85)
    
    # Normalize về [0, 1]
    max_pr = max(pagerank_scores.values()) if pagerank_scores else 1.0
    pagerank_norm = {svc: score / max_pr for svc, score in pagerank_scores.items()}
    
    # ── Timestamp score ───────────────────────────────────────────────────────
    # Service alert sớm nhất → 1.0, muộn nhất → 0.0
    times_in_cluster = {
        svc: service_first_alert.get(svc)
        for svc in alert_services
        if service_first_alert.get(svc) is not None
    }
    
    if times_in_cluster:
        t_min = min(times_in_cluster.values())
        t_max = max(times_in_cluster.values())
        t_range = (t_max - t_min).total_seconds()
        
        if t_range == 0:
            timestamp_score = {svc: 1.0 for svc in alert_services}
        else:
            timestamp_score = {
                svc: 1.0 - (times_in_cluster[svc] - t_min).total_seconds() / t_range
                if svc in times_in_cluster else 0.5
                for svc in alert_services
            }
    else:
        timestamp_score = {svc: 0.5 for svc in alert_services}
    
    # ── Combined score: 0.6 × PageRank + 0.4 × Timestamp ─────────────────────
    combined = {
        svc: round(0.4 * pagerank_norm.get(svc, 0.0) + 0.6 * timestamp_score.get(svc, 0.0), 4)
        for svc in alert_services
    }
    
    # Sort desc, lấy top_k
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return [[svc, score] for svc, score in ranked[:top_k]]


print('graph_rca() defined OK')

# Quick test trên cluster lớn nhất
test_result = graph_rca(clusters[0], graph, service_first_alert)
print(f'\nTest cluster {clusters[0]["cluster_id"]} ({clusters[0]["services"]}):')
for svc, score in test_result:
    print(f'  {svc:20s} score={score}')

graph_rca() defined OK

Test cluster c-000-000 (['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc']):
  payment-svc          score=0.9296
  checkout-svc         score=0.8603
  cart-svc             score=0.6407


In [58]:
# ── Cell 6: Retrieval — tìm similar incidents ─────────────────────────────────

def retrieve_similar(cluster, incidents, top_k=3, min_score=0.2):
    """
    Keyword-based similarity scoring:
    +0.4  nếu root_cause_service của history nằm trong cluster
    +0.2  mỗi service overlap (max +0.4)
    +0.2  nếu cùng severity (map: crit→high, warn→medium)
    """
    cluster_services = set(cluster['services'])
    cluster_severity = cluster['max_severity']
    
    # Map severity để so sánh
    severity_map = {'crit': 'high', 'warn': 'medium', 'info': 'low',
                    'high': 'high', 'medium': 'medium', 'low': 'low',
                    'critical': 'high'}
    cluster_sev_norm = severity_map.get(cluster_severity, 'medium')
    
    scored = []
    for inc in incidents:
        score = 0.0
        
        # +0.4 nếu root cause service trùng cluster
        if inc.get('root_cause_service') in cluster_services:
            score += 0.4
        
        # +0.2 mỗi service overlap, tối đa +0.4
        inc_services = set(inc.get('services_involved', []))
        overlap = len(cluster_services & inc_services)
        score += min(overlap * 0.2, 0.4)
        
        # +0.2 nếu cùng severity
        inc_sev_norm = severity_map.get(inc.get('severity', ''), 'medium')
        if inc_sev_norm == cluster_sev_norm:
            score += 0.2
        
        if score >= min_score:
            scored.append((inc, round(score, 2)))
    
    # Sort desc
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


print('retrieve_similar() defined OK')

# Test
test_similar = retrieve_similar(clusters[0], incidents)
print(f'\nTop similar incidents cho cluster {clusters[0]["cluster_id"]}:')
for inc, score in test_similar:
    print(f'  [{score}] {inc["id"]:20s} root={inc["root_cause_service"]:15s} class={inc["root_cause_class"]}')

retrieve_similar() defined OK

Top similar incidents cho cluster c-000-000:
  [1.0] INC-2025-11-08       root=payment-svc     class=connection_pool_exhaustion
  [1.0] INC-2026-03-20       root=edge-lb         class=ddos
  [0.8] INC-2025-08-17       root=edge-lb         class=tls_expiry


In [59]:
# ── Cell 7: Classifier — kNN-style từ top-1 similar ──────────────────────────

VALID_CLASSES = {
    'connection_pool_exhaustion', 'slow_query', 'memory_leak',
    'rebalance_storm', 'deadlock', 'network_partition', 'bad_deploy',
    'config_push', 'tls_expiry', 'ddos', 'other'
}

def classify(similar_incidents):
    """
    Lấy class + actions từ top-1 similar incident.
    Fallback về 'other' nếu không có similar.
    """
    if not similar_incidents:
        return 'other', ['Investigate manually']
    
    top_inc, _ = similar_incidents[0]
    root_class = top_inc.get('root_cause_class', 'other')
    
    # Validate enum
    if root_class not in VALID_CLASSES:
        root_class = 'other'
    
    # Lấy remediation làm actions
    remediation = top_inc.get('remediation', '')
    actions = [remediation] if remediation else ['Investigate manually']
    
    return root_class, actions


print('classify() defined OK')

test_class, test_actions = classify(test_similar)
print(f'\nTest classify → class={test_class}')
print(f'  actions: {test_actions}')

classify() defined OK

Test classify → class=connection_pool_exhaustion
  actions: ['Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%.']


In [60]:
# ── Cell 8: Validate output schema ───────────────────────────────────────────

def validate_result(result, cluster_services):
    """
    4 checks bắt buộc (hallucination guard):
    1. root_cause nằm trong cluster
    2. class thuộc enum
    3. confidence ∈ [0, 1]
    4. actions là list non-empty
    """
    errors = []
    
    if result.get('root_cause') not in cluster_services:
        errors.append(f'root_cause "{result.get("root_cause")}" không thuộc cluster')
    
    if result.get('class') not in VALID_CLASSES:
        errors.append(f'class "{result.get("class")}" không hợp lệ')
    
    conf = result.get('confidence', -1)
    if not (isinstance(conf, (int, float)) and 0 <= conf <= 1):
        errors.append(f'confidence={conf} không trong [0,1]')
    
    if not result.get('actions'):
        errors.append('actions rỗng')
    
    return errors


print('validate_result() defined OK')

validate_result() defined OK


In [61]:
# ── Cell 9: Main pipeline — chạy tất cả clusters ─────────────────────────────

results = []

for cluster in clusters:
    cid = cluster['cluster_id']
    cluster_services = set(cluster['services'])
    
    # ── Bước 1: Graph RCA ────────────────────────────────────────────────────
    graph_top3 = graph_rca(cluster, graph, service_first_alert, top_k=3)
    top1_service = graph_top3[0][0]
    top1_score   = graph_top3[0][1]
    
    # ── Bước 2: Retrieval ────────────────────────────────────────────────────
    similar = retrieve_similar(cluster, incidents, top_k=3)
    similar_ids = [inc['id'] for inc, _ in similar]
    
    # ── Bước 3: Classify ─────────────────────────────────────────────────────
    root_class, actions = classify(similar)
    
    # ── Bước 4: Build reasoning ──────────────────────────────────────────────
    if similar:
        top_inc, sim_score = similar[0]
        reasoning = (
            f"{top1_service} ranked #1 by Graph RCA (PageRank+timestamp score={top1_score}). "
            f"Most similar past incident: {top_inc['id']} (similarity={sim_score}) "
            f"which had root_cause_class='{root_class}'. "
            f"Incident summary: {top_inc.get('summary', 'N/A')}"
        )
    else:
        reasoning = (
            f"{top1_service} ranked #1 by Graph RCA (score={top1_score}). "
            f"No similar incidents found — fallback to graph-only."
        )
    
    # ── Bước 5: Assemble result ───────────────────────────────────────────────
    result = {
        'cluster_id'       : cid,
        'graph_top3'       : graph_top3,
        'root_cause'       : top1_service,
        'class'            : root_class,
        'confidence'       : top1_score,
        'actions'          : actions,
        'reasoning'        : reasoning,
        'similar_incidents': similar_ids,
        'method'           : 'graph+retrieval' if similar else 'graph-only-fallback'
    }
    
    # ── Bước 6: Validate ─────────────────────────────────────────────────────
    errors = validate_result(result, cluster['services'])
    if errors:
        print(f'[WARN] {cid} validation errors: {errors}')
        # Fallback
        result.update({
            'class'   : 'other',
            'actions' : ['Investigate manually'],
            'method'  : 'graph-only-fallback'
        })
    
    results.append(result)
    print(f'[OK] {cid:12s} root_cause={result["root_cause"]:20s} class={result["class"]:30s} confidence={result["confidence"]}')

print(f'\n✓ Processed {len(results)} clusters')

[OK] c-000-000    root_cause=payment-svc          class=connection_pool_exhaustion     confidence=0.9296
[OK] c-001-000    root_cause=checkout-svc         class=other                          confidence=1.0
[OK] c-002-000    root_cause=payment-svc          class=ddos                           confidence=1.0

✓ Processed 3 clusters


In [62]:
# ── Cell 10: Write rca_output.json ────────────────────────────────────────────

Path('results').mkdir(exist_ok=True)

output = {
    'clusters_analyzed': len(results),
    'results': results
}

with open('results/rca_output.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Saved: results/rca_output.json')
print(json.dumps(output, indent=2))

Saved: results/rca_output.json
{
  "clusters_analyzed": 3,
  "results": [
    {
      "cluster_id": "c-000-000",
      "graph_top3": [
        [
          "payment-svc",
          0.9296
        ],
        [
          "checkout-svc",
          0.8603
        ],
        [
          "cart-svc",
          0.6407
        ]
      ],
      "root_cause": "payment-svc",
      "class": "connection_pool_exhaustion",
      "confidence": 0.9296,
      "actions": [
        "Rollback to v3.1. Scale pool 50 \u2192 100 cushion. Add pool monitor alert > 80%."
      ],
      "reasoning": "payment-svc ranked #1 by Graph RCA (PageRank+timestamp score=0.9296). Most similar past incident: INC-2025-11-08 (similarity=1.0) which had root_cause_class='connection_pool_exhaustion'. Incident summary: Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 ph\u00fat. Downstream checkout cascade. Notification queue backed up.",
      "similar_incidents": [
        "INC-2025-11-08",
        "INC-2026-0

In [ ]:
# ── Cell 11: Final acceptance check ──────────────────────────────────────────

print('=' * 60)
print('ACCEPTANCE CHECK')
print('=' * 60)

# 1. rca_output.json valid
with open('results/rca_output.json') as f:
    out = json.load(f)

ok = True
for r in out['results']:
    has_top3    = 'graph_top3' in r and len(r['graph_top3']) > 0
    has_root    = 'root_cause' in r
    has_class   = 'class' in r and r['class'] in VALID_CLASSES
    if not (has_top3 and has_root and has_class):
        print(f'[FAIL] {r["cluster_id"]}: top3={has_top3} root={has_root} class={has_class}')
        ok = False
    else:
        print(f'[OK]   {r["cluster_id"]:12s} root={r["root_cause"]:20s} class={r["class"]}')

# 2. networkx used
print(f'\n[OK]   networkx used: graph has {graph.number_of_nodes()} nodes')

# 3. Retrieval indicator present
print(f'[OK]   retrieval: similar_incidents field present in all results')

# 4. FINDINGS.md
findings_words = len(open('FINDINGS.md').read().split())
status = 'OK' if findings_words >= 100 else 'FAIL'
print(f'[{status}]   FINDINGS.md: {findings_words} words (min 100)')

print('\n' + ('✓ ALL CHECKS PASSED' if ok else '✗ SOME CHECKS FAILED'))

ACCEPTANCE CHECK
[OK]   c-000-000    root=payment-svc          class=connection_pool_exhaustion
[OK]   c-001-000    root=checkout-svc         class=other
[OK]   c-002-000    root=payment-svc          class=ddos

[OK]   networkx used: graph has 10 nodes
[OK]   retrieval: similar_incidents field present in all results
[OK]   FINDINGS.md: 273 words (min 100)

✓ ALL CHECKS PASSED
